# 02 · LangGraph pilot

A few **development** claims through all four approaches, showing each graph's shape, the node-by-node execution trace, the evidence retrieved, the deterministic numeric checks and the initial vs final verdict.

**`MODE` defaults to `mock`** (keyword stand-in, no credentials, not results). `offline_cache` uses stored model responses only. `live` makes paid Claude calls and needs `APPROVAL_PHRASE` set to the exact phrase the cost plan prints. Gold claims are never used here.

In [ ]:
import os, sys, json
from dataclasses import asdict
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("research root:", ROOT)

In [ ]:
CONFIG = "configs/main_comparison.json"
PILOT_CLAIMS = ["DEV-AMZN-06", "DEV-AAPL-04", "DEV-MSFT-02"]
MODE = "mock"                  # mock | offline_cache | live
APPROVAL_PHRASE = ""           # live only

## Graph shapes

In [ ]:
from graphs import build_graph, mermaid, GRAPH_VERSION
from engine.config import load_config
cfg = load_config(CONFIG)
print("graph version:", GRAPH_VERSION)
for name, a in cfg["approaches"].items():
    g = build_graph(a["kind"], bool(a["contradiction"]["enabled"]) if a["kind"] == "contradiction_aware" else True).get_graph()
    edges = [f"{e.source}->{e.target}" + (" (conditional)" if e.conditional else "") for e in g.edges]
    print(f"{name} [{a['kind']}]\n   " + "\n   ".join(edges))
print("\nMermaid source for A3 (paste into any Mermaid viewer):\n")
print(mermaid("contradiction_aware"))

In [ ]:
from engine import plots
from engine.notebook import save_figure
plots.style()
specs = [("conventional_rag", "A1 Conventional RAG"), ("financial_verification", "A2 RAG + numeric verification"),
         ("contradiction_aware", "A3 Contradiction-aware (proposed)"), ("self_reflection", "SR Self-reflection (control)")]
fig = plots.graph_diagram([(k, t, build_graph(k)) for k, t in specs],
                          "LangGraph StateGraphs of the four claim-verification approaches")
print("saved", save_figure(fig, "langgraph_architecture")); plt.show()

## Run the pilot

In [ ]:
from engine.runner import select_claims, plan_run, run_experiment, load_run
from engine.llm import approve_paid_run
from engine.notebook import banner, describe_run, trace_index
claims, info = select_claims(cfg, split="development", claim_ids=PILOT_CLAIMS)
plan = plan_run(cfg, claims)
print(f"upper bound: {plan['max_llm_calls']} calls, ${plan['max_cost_usd']:.2f}")
if MODE == "live":
    run = load_run(run_experiment(cfg, claims, info, llm_mode="live", approval=approve_paid_run(plan, APPROVAL_PHRASE)))
elif MODE == "offline_cache":
    run = load_run(run_experiment(cfg, claims, info, llm_mode="offline"))
else:
    run = load_run(run_experiment(cfg, claims, info, llm_mode="mock", progress=lambda s: None))
describe_run(run)
traces = trace_index(run)

## Execution trace (node by node)

In [ ]:
rows = []
for (approach, cid), t in sorted(traces.items()):
    for i, s in enumerate((t.get("graph") or {}).get("execution_trace", [])):
        rows.append({"approach": approach, "claim_id": cid, "step": i + 1, "node": s["node"], "status": s["status"],
                     "ms": s["duration_ms"], "llm_calls": s["llm_calls_so_far"], "queries": s["retrieval_queries_so_far"],
                     "detail": {k: v for k, v in s.items() if k not in ("node", "status", "duration_ms", "llm_calls_so_far",
                                                                         "retrieval_queries_so_far")}})
steps = pd.DataFrame(rows)
steps[steps.claim_id == PILOT_CLAIMS[0]]

## Retrieved evidence and numeric checks (A3, first pilot claim)

In [ ]:
t = traces[("A3_contradiction_aware", PILOT_CLAIMS[0])]
print("claim:", t["claim"]["claim"], "| cutoff:", t["claim"]["as_of_date"])
display(pd.DataFrame(t["corpus"]["filings"])[["form", "accession_number", "filed", "report_date"]])
ev = pd.DataFrame(t["evidence"])
display(ev[["evidence_id", "kind", "form", "filed", "period_end", "locator"]].head(15))
checks = [c for s in t["tool_calls"] if s["tool"] == "verify_numbers" for c in s["checks"]]
display(pd.DataFrame([{"metric": c["assertion"].get("metric"), "measure": c["assertion"].get("measure"),
                       "basis": c["assertion"].get("period_basis"), "status": c["status"], "issues": c["issues"],
                       "period": (c["computed"] or {}).get("period_end"), "value": (c["computed"] or {}).get("level"),
                       "yoy": (c["computed"] or {}).get("yoy_change")} for c in checks]))

## Initial vs final verdicts

In [ ]:
preds = pd.DataFrame(run["predictions"])
cols = ["approach", "claim_id", "initial_label", "predicted_label", "verdict_changed", "stop_reason", "llm_calls",
        "retrieval_queries", "numeric_checks"]
display(preds[cols])
recs = [{"approach": p["approach"], "claim_id": p["claim_id"], **p["second_look_record"]}
        for p in run["predictions"] if p.get("second_look_record")]
pd.DataFrame(recs)

## Data isolation checks

In [ ]:
from engine.metrics import integrity_metrics
integ = pd.DataFrame(integrity_metrics(run["predictions"], traces))
display(integ)
assert (integ.future_data_leakage_rate == 0).all(), "a run used data filed after its cutoff"
assert set(run["manifest"]["dataset"]["claim_ids"]) <= set(PILOT_CLAIMS) and run["manifest"]["dataset"]["split"] == "development"
print("OK: development claims only; nothing filed after any cutoff was used")